# Chapter 8: COD spikes and the beer behind them

Turns a daily effluent series (volume and COD) plus the beer packaged that day into a list of spike days and the hL of beer each would imply.

Baseline: the median kg of COD per hL packaged, times the day's volume. Kestrel Lager carries about 13 kg of COD a hL (chapter 8.2). A spike is a lead to follow, not proof: cleaning chemicals, grain carry-over and yeast raise COD too.

Synthetic Kestrel Brewing data. No real brewery's numbers. Book rates: USD 1 = INR 90; EUR 1 = USD 1.10.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

e = pd.read_csv("../data/effluent_daily.csv", parse_dates=["date"])
e["cod_kg"] = e.effluent_m3 * e.cod_mg_l / 1000
ratio = (e.cod_kg / e.packaged_hl).median()
e["baseline_kg"] = ratio * e.packaged_hl
e["excess_kg"] = e.cod_kg - e.baseline_kg
print(f"Baseline {ratio:.3f} kg of COD a hL packaged; water to effluent {(e.effluent_m3 / e.packaged_hl).median():.2f} m3 a hL")

In [ ]:
KG_COD_PER_HL_BEER = 13
noise = 1.4826 * (e.excess_kg - e.excess_kg.median()).abs().median()
threshold = max(5 * noise, 50)
spikes = e[e.excess_kg > threshold].copy()
spikes["beer_hl_implied"] = (spikes.excess_kg / KG_COD_PER_HL_BEER).round(0)
spikes["usd_replacement"] = spikes.beer_hl_implied * 32
spikes["usd_upper_bound"] = spikes.beer_hl_implied * 112
print(f"Spike threshold {threshold:.0f} kg above baseline")
print(spikes[["date", "excess_kg", "beer_hl_implied", "usd_replacement", "usd_upper_bound"]].round(0).to_string(index=False))
spikes[["date", "packaged_hl", "cod_kg", "excess_kg", "beer_hl_implied", "usd_replacement", "usd_upper_bound"]].round(0)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3.5))
ax.plot(e.date, e.cod_kg, color="#0b4f45", label="COD load, kg a day")
ax.plot(e.date, e.baseline_kg, color="grey", ls="--", label="Baseline from volume")
ax.scatter(spikes.date, spikes.cod_kg, color="#b23a2e", zorder=3, label="Spike day")
for _, r in spikes.iterrows():
    ax.annotate(f"{r.beer_hl_implied:.0f} hL", (r.date, r.cod_kg), textcoords="offset points", xytext=(0, 6), ha="center")
ax.set_ylabel("kg COD a day")
ax.set_title("Effluent COD against baseline (synthetic)")
ax.legend(loc="upper left")
plt.tight_layout()
plt.show()

**Next step.** Take each spike day to the shift log. What was rinsed, dumped or disconnected? Only a person can say it was beer.